### Yield curve test

In [ ]:
import torch
import torch.nn as nn
from torchdiffeq import *
import matplotlib.pyplot as plt
import numpy as np
from solvers import *
import pandas as pd
import re

torch.manual_seed(0)
device = torch.device('cpu')  # change to 'cuda' if you have a GPU
dtype = torch.float64

In [ ]:
def tenor_to_years(tenor: str) -> float:
    """
    Parse tenors like '1D','2W','3M','6M','1Y','2Y','10Y' into year fractions.
    Uses simple conversions: 365d, 52w, 12m.
    """
    t = str(tenor).strip().upper()
    m = re.fullmatch(r"(\d+)\s*([DWMY])", t)
    if not m:
        raise ValueError(f"Could not parse tenor: {tenor}")
    n = int(m.group(1))
    u = m.group(2)
    if u == "D":
        return n / 365.0
    if u == "W":
        return n / 52.0
    if u == "M":
        return n / 12.0
    if u == "Y":
        return float(n)
    raise ValueError(f"Unknown tenor unit: {u}")

In [ ]:
class Swap:
    def __init__(self, tenor: str, rate: float | None = None):
        self.tenor_days = Swap.parse_tenor(tenor)
        self.rate = rate
        self.maturity = self.tenor_days / 360.0
        if self.tenor_days < 720:
            self.frequency = self.maturity
        else:
            self.frequency = 0.5  # semi-annual payments for tenors > 2 years
        self.yfs = torch.arange(0, self.maturity+self.frequency, self.frequency)

    def fair_rate(self, model: nn.Module) -> torch.Tensor:
        discounts = model.discounts(self.yfs)[1:]
        pv01 = torch.sum(discounts * self.frequency)
        fixed_leg = 1.0 - discounts[-1]
        fair_rate = fixed_leg / pv01
        return fair_rate.squeeze()
    
    @staticmethod
    def parse_tenor(tenor: str) -> int:
        if tenor.endswith('d'):
            return int(tenor[:-1])
        elif tenor.endswith('y'):
            return int(tenor[:-1])*360
        else:
            raise ValueError("Unknown tenor format")

In [ ]:
market_rates_df = pd.read_excel("swaps.xlsx", index_col='date', parse_dates=['date'])
market_rates_df = market_rates_df.sort_index().dropna()
market_rates_df /= 100  
tenors = np.array([tenor_to_years(col) for col in market_rates_df.columns])

In [ ]:
class ODEFunc(nn.Module):
    def __init__(self):
        super(ODEFunc, self).__init__()
        self.net = nn.Sequential(
            # Input dimension is 2, hidden layer size is 50
            nn.Linear(2, 5, dtype=torch.float64),
            nn.Tanh(),          # Activation function (Tanh)
            nn.Linear(5, 1, dtype=torch.float64)    # Output dimension is 2
        )

    def forward(self, t, y):
        x = torch.stack([t, y.squeeze()])  # Concatenate time and state
        # Forward pass: returns the time derivative of the state
        return self.net(x)

    def discounts(self, t):
        # latent initial value
        y0 = torch.tensor(1.0, dtype=torch.float64)
        # integrate latent y(t)
        P = odeint(self, y0, t, method='rk4')
        return P
    
    def fwd_rates(self, t, T):
        discounts_t = self.discounts(t)
        discounts_T = self.discounts(T)
        fwd_rates = (discounts_t / discounts_T - 1) / (T - t)
        return fwd_rates

In [ ]:
ref_date = market_rates_df.index[-1]
swap_rates_mkt_np = market_rates_df.iloc[-1].values
swaps = [Swap(tenor) for tenor in market_rates_df.columns]
model = ODEFunc().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
swap_rates_mkt = torch.tensor(swap_rates_mkt_np, dtype=torch.float64)

In [ ]:
for swap in swaps:
    print(swap.yfs)

In [ ]:
# def train_curve(model: nn.Module, swaps: list[Swap], target_swaps: torch.Tensor, steps=10000, print_every=10):
#     for epoch in range(1, steps + 1):
#         optimizer.zero_grad()
#         preds = torch.stack([s.fair_rate(model) for s in swaps])  # (M,)
#         loss = criterion(preds, target_swaps)
#         loss.backward()
#         optimizer.step()
#         if epoch % print_every == 0:
#             with torch.no_grad():
#                 mae = torch.mean(torch.abs(preds - target_swaps)).item()
#             print(f"Epoch {epoch:4d} | MSE {loss.item():.6e} | MAE {mae:.6e}")


# train_curve(model, swaps, swap_rates_mkt, steps=10000, print_every=10)

In [ ]:
optimizer = SSBroyden(
    model.parameters(),    
    max_iter=2000,          # do real work
    tolerance_change=1e-15,
    tolerance_grad=1e-15,
)

def closure(*args, **kwargs):
    optimizer.zero_grad()
    preds = torch.stack([s.fair_rate(model) for s in swaps])  # (M,)
    loss = criterion(preds, swap_rates_mkt)
    loss.backward()
    print(f"Loss: {loss.item():.6e}")
    return loss

loss = optimizer.step(closure)

In [ ]:
# --- quick check: print market vs model --------------------------------------
# Define maturities and helper function
maturities = np.array([s.maturity for s in swaps], dtype=float)
swap_rates_mkt_np = swap_rates_mkt.cpu().numpy()
swaps_rates_from_model = [s.fair_rate(model).detach().cpu().numpy() for s in swaps]

print("\nMaturity  Market     Model")
for m, sm, sp in zip(maturities, swap_rates_mkt_np, swaps_rates_from_model):
    print(f"{m:>7.1f}  {sm:>7.4f}  {sp:>7.4f}")

In [ ]:
plt.plot(maturities, swap_rates_mkt_np, 'o-', label='Market')
plt.plot(maturities, swaps_rates_from_model, 'x--', label='Model')
plt.xlabel('Maturity (years)')
plt.ylabel('Par Swap Rate')
plt.title('Market vs Model Par Swap Rates')
plt.legend()
plt.show()

## Extrapolation test

In [ ]:
swap_tenors = ['1d'] + [str(i*30)+'d' for i in range(1,13)] + ['540d'] + [str(i)+'y' for i in range(2,21)]
swaps_extrapolation = [Swap(tenor) for tenor in swap_tenors]
maturities_extrap = np.array([s.maturity for s in swaps_extrapolation], dtype=float)

with torch.no_grad():
    pred_swaps_extrap = torch.stack([s.fair_rate(model) for s in swaps_extrapolation])

print("\nExtrapolation Test")
print("Maturity  Model")
for m, sp in zip(maturities_extrap, pred_swaps_extrap):
    print(f"{m:>7.1f}  {sp:>7.4f}")

plt.plot(maturities, swap_rates_mkt_np, 'o-', label='Market')
plt.plot(maturities_extrap, pred_swaps_extrap, '--', label='Model (Extrapolated)')
plt.xlabel('Maturity (years)')
plt.ylabel('Par Swap Rate')
plt.title('Market vs Model Par Swap Rates with Extrapolation')
plt.legend()
plt.show()

In [ ]:
def fwd_rates(dfs: np.ndarray, times: np.ndarray) -> np.ndarray:
    """
    Continuously-compounded forward rates over each interval [t_i, t_{i+1}].
    More stable than simple forwards from DF ratios.
    """
    dfs = np.asarray(dfs, dtype=float)
    times = np.asarray(times, dtype=float)

    # safety
    if len(dfs) != len(times):
        raise ValueError("dfs and times must have same length")
    if np.any(dfs <= 0):
        raise ValueError("discount factors must be positive")
    dt = np.diff(times)
    if np.any(dt <= 0):
        raise ValueError("times must be strictly increasing")

    return -np.diff(np.log(dfs)) / dt

def inst_fwd_from_logdf(dfs: np.ndarray, times: np.ndarray) -> np.ndarray:
    """
    Approximate instantaneous forward f(t) = -d/dt ln DF(t) using central differences.
    Returns f at interior points times[1:-1].
    """
    dfs = np.asarray(dfs, float)
    times = np.asarray(times, float)
    logdf = np.log(dfs)

    f = np.zeros(len(times) - 2)
    for i in range(1, len(times) - 1):
        f[i-1] = -(logdf[i+1] - logdf[i-1]) / (times[i+1] - times[i-1])
    return f, times[1:-1]


times = torch.arange(0, 10.0 + 0.1, 0.1, dtype=torch.float64)
dfs = model.discounts(times)

times = times.detach().cpu().numpy()
dfs = dfs.detach().cpu().numpy()
forward_rates = fwd_rates(dfs, times)

plt.plot(times[:-1], forward_rates, label='Model Forward Rates')
plt.xlabel('Maturity (years)')
plt.ylabel('Forward Rate')
plt.title('Model Implied Forward Rates')
plt.legend()
plt.show()

In [ ]:
class ODEFunc(nn.Module):
    def __init__(self, n_tenors: int):
        super(ODEFunc, self).__init__()
        self.net = nn.Sequential(
            # Input dimension is 2, hidden layer size is 50
            nn.Linear(n_tenors+2, 32, dtype=torch.float64),
            nn.Tanh(),          # Activation function (Tanh)
            nn.Linear(32, 1, dtype=torch.float64)    # Output dimension is 2
        )
    
    def set_swap_rates(self, rates: torch.Tensor):
        self.swap_rates = rates

    def forward(self, t, y):
        if t.dim() == 0:
            t = t.unsqueeze(0)
        x = torch.cat([t, self.swap_rates, y])
        return self.net(x)

    def discounts(self, t):
        # latent initial value
        y0 = torch.tensor([1.0], dtype=torch.float64)
        # integrate latent y(t)
        P = odeint(self, y0, t, method='rk4')
        return P
    
    def fwd_rates(self, t, T):
        discounts_t = self.discounts(t)
        discounts_T = self.discounts(T)
        fwd_rates = (discounts_t / discounts_T - 1) / (T - t)
        return fwd_rates

In [ ]:
model = ODEFunc(n_tenors=len(tenors)).to(device)

In [ ]:
class ODEFuncDeepONet(nn.Module):
    """
    DeepONet-style ODE function:
      branch(rates) -> b in R^p
      trunk(t,y)    -> h in R^p
      f(t,y; rates) = <b, h> + bias
    """
    def __init__(self, n_tenors: int, p: int = 8, hidden: int = 16, dtype=torch.float64):
        super().__init__()
        self.dtype = dtype
        # Small branch net: rates -> features (p,)
        self.branch = nn.Sequential(
            nn.Linear(n_tenors, hidden, dtype=dtype),
            nn.Tanh(),
            nn.Linear(hidden, p, dtype=dtype),
        )

        # Small trunk net: (t, y) -> features (p,)
        self.trunk = nn.Sequential(
            nn.Linear(2, hidden, dtype=dtype),
            nn.Tanh(),
            nn.Linear(hidden, p, dtype=dtype),
        )

        self.bias = nn.Parameter(torch.zeros((), dtype=dtype))

    def set_swap_rates(self, rates: torch.Tensor):
        # copy into buffer to preserve device/dtype
        self.swap_rates = rates

    def forward(self, t, y):
        # odeint passes scalar t and state y with shape like (1,) or ()
        t = t.reshape(1)
        y = y.reshape(1)

        # trunk input is (t, y)
        ty = torch.stack([t.squeeze(0), y.squeeze(0)]).unsqueeze(0)  # (1, 2)

        # DeepONet pieces
        b = self.branch(self.swap_rates.unsqueeze(0))  # (1, p)
        h = self.trunk(ty)                              # (1, p)

        out = (b * h).sum(dim=-1) + self.bias          # (1,)
        return out.reshape_as(y)                        # match shape of y for odeint

    def discounts(self, t):
        # Make sure t is on same device/dtype
        t = t.to(dtype=self.swap_rates.dtype, device=self.swap_rates.device)
        y0 = torch.tensor([1.0], dtype=self.swap_rates.dtype, device=self.swap_rates.device)
        P = odeint(self, y0, t, method="rk4")  # (len(t), 1)
        return P

    def fwd_rates(self, t, T):
        discounts_t = self.discounts(t)[1:]
        discounts_T = self.discounts(T)[1:]
        return (discounts_t / discounts_T - 1) / (T - t)

In [ ]:
model = ODEFuncDeepONet(n_tenors=len(tenors)).to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
def train_curve(model: nn.Module, market_rates: pd.DataFrame, steps=10000):
    for epoch in range(1, steps + 1):
        optimizer.zero_grad()
        loss = 0.0
        for _, rates in market_rates.iterrows():
            obs_swap_rates = torch.tensor(rates.values, dtype=torch.float64)
            swaps = [Swap(tenor) for tenor in market_rates.columns]
            model.set_swap_rates(obs_swap_rates)
            preds = torch.stack([s.fair_rate(model) for s in swaps])
            if preds.shape != obs_swap_rates.shape:
                raise ValueError("Shape mismatch between predictions and observations")
            
            loss += criterion(preds, obs_swap_rates)*100
            
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch:4d} | MSE {loss.item():.6e}")

traning_curves = market_rates_df.sample(n=50)
train_curve(model, traning_curves, steps=10_000)

In [ ]:
optimizer = SSBroyden(
    model.parameters(),
    max_iter=2000,          # do real work
    tolerance_change=1e-15,
    tolerance_grad=1e-15,
)

def train_curve(optimizer, model: nn.Module, market_rates: pd.DataFrame):
    def closure(*args, **kwargs):
        optimizer.zero_grad()
        loss = 0.0
        for _, rates in market_rates.iterrows():
            obs_swap_rates = torch.tensor(rates.values, dtype=torch.float64)
            swaps = [Swap(tenor) for tenor in market_rates.columns]
            model.set_swap_rates(obs_swap_rates)
            preds = torch.stack([s.fair_rate(model) for s in swaps])
            if preds.shape != obs_swap_rates.shape:
                raise ValueError("Shape mismatch between predictions and observations")
            
            loss += criterion(preds, obs_swap_rates)*100
            
        loss.backward()
        print(f"Loss: {loss.item():.6e}")
        return loss
    
    optimizer.step(closure)

traning_curves = market_rates_df.sample(n=50, random_state=2)
train_curve(optimizer, model, traning_curves)

In [ ]:
# --- quick check: print market vs model --------------------------------------
# Define maturities and helper function
swap_rates_mkt = torch.tensor(market_rates_df.iloc[-250].values, dtype=torch.float64)
model.set_swap_rates(swap_rates_mkt)

maturities = np.array([s.maturity for s in swaps], dtype=float)
swap_rates_mkt_np = swap_rates_mkt.cpu().numpy()
swaps_rates_from_model = [s.fair_rate(model).detach().cpu().numpy() for s in swaps]

print("\nMaturity  Market     Model")
for m, sm, sp in zip(maturities, swap_rates_mkt_np, swaps_rates_from_model):
    print(f"{m:>7.1f}  {sm:>7.4f}  {sp.item():>7.4f}")

plt.plot(maturities, swap_rates_mkt_np, 'o-', label='Market')
plt.plot(maturities, swaps_rates_from_model, 'x--', label='Model')
plt.xlabel('Maturity (years)')
plt.ylabel('Par Swap Rate')
plt.title('Market vs Model Par Swap Rates')
plt.legend()
plt.show()

# forward rates

times = torch.arange(0, 20.0 + 0.1, 0.1, dtype=torch.float64)
dfs = model.discounts(times).squeeze(-1)

dt = times[1:] - times[:-1]
fwds = (dfs[:-1] / dfs[1:] - 1) / dt

plt.figure(figsize=(10, 6))
plt.plot(times[:-1].detach().cpu().numpy(), fwds.detach().cpu().numpy(), label='Model Forward Rates')
plt.xlabel('Maturity (years)')
plt.ylabel('Forward Rate')
plt.title('Model Implied Forward Rates')
plt.legend()
plt.show()

In [ ]:
model = ODEFuncDeepONet(n_tenors=len(tenors)).to(device)

In [ ]:
optimizer = SSBroyden(
    model.parameters(),
    max_iter=2000,          # do real work
    tolerance_change=1e-15,
    tolerance_grad=1e-15,
)


def _make_adjacent_date_pairs(df: pd.DataFrame, n_pairs: int, seed: int | None = None):
    """
    Returns list of (prev_row, next_row, prev_idx, next_idx) pairs where next is one step after prev.
    Samples n_pairs distinct 'next' indices from [1, len(df)-1].
    """
    if len(df) < 2:
        raise ValueError("Need at least 2 rows to form adjacent pairs.")

    rng = np.random.default_rng(seed)
    max_pairs = len(df) - 1
    n_pairs = min(n_pairs, max_pairs)

    next_ix = rng.choice(np.arange(1, len(df)), size=n_pairs, replace=False)
    prev_ix = next_ix - 1

    # sort by time (optional, but helps reproducibility/debugging)
    order = np.argsort(next_ix)
    next_ix, prev_ix = next_ix[order], prev_ix[order]

    pairs = []
    for pi, ni in zip(prev_ix, next_ix):
        pairs.append((df.iloc[pi], df.iloc[ni], pi, ni))
    return pairs


def train_curve(
    optimizer,
    model: nn.Module,
    market_rates: pd.DataFrame,
    pillars: torch.Tensor,
    n_samples: int = 50,
    seed: int | None = None,
    fit_weight: float = 100.0,
    time_weight: float = 0.01,
):
    """
    Trains using SSBroyden closure on:
      - swap rate fit at time t (next row)
      - forward consistency between (t-1) and t on given pillars
    Uses sampled adjacent pairs for diversity.
    """
    if not isinstance(market_rates.index, pd.DatetimeIndex):
        # not required, but your code assumes time ordering
        market_rates = market_rates.copy()

    market_rates = market_rates.sort_index()

    # Build once
    swaps = [Swap(tenor) for tenor in market_rates.columns]
    pairs = _make_adjacent_date_pairs(market_rates, n_pairs=n_samples, seed=seed)

    pair_tensors = []
    for prev_row, next_row, _, _ in pairs:
        r_prev = torch.tensor(prev_row.values, dtype=dtype, device=device)
        r_next = torch.tensor(next_row.values, dtype=dtype, device=device)
        pair_tensors.append((r_prev, r_next))

    # Forward pillars to device/dtype once
    pillars = pillars.to(device=device, dtype=dtype)
 

    def closure(*args, **kwargs):
        optimizer.zero_grad()
        total_loss = torch.zeros((), dtype=dtype, device=device)

        for r_prev, r_next in pair_tensors:
            # ---- fit to swaps at "next" time ----
            model.set_swap_rates(r_next)
            preds = torch.stack([s.fair_rate(model) for s in swaps])
            
            total_loss = total_loss + fit_weight * criterion(preds, r_next)

            # ---- consistency between prev and next dfs----
            dfs_next = model.discounts(pillars).squeeze()

            model.set_swap_rates(r_prev)
            dfs_prev = model.discounts(pillars).squeeze()

            total_loss = total_loss + time_weight * criterion(dfs_next, dfs_prev)

        total_loss.backward()
        print(f"Loss: {total_loss.item():.6e}")
        return total_loss

    optimizer.step(closure)

train_curve(
    optimizer,
    model,
    market_rates_df,
    pillars=torch.arange(0.0, 10.0 + 1.0, 1.0, dtype=torch.float64),
    n_samples=50,
    seed=123,
)

In [ ]:
# --- quick check: print market vs model --------------------------------------
# Define maturities and helper function
swap_rates_mkt = torch.tensor(market_rates_df.iloc[-100].values, dtype=torch.float64)
model.set_swap_rates(swap_rates_mkt)

maturities = np.array([s.maturity for s in swaps], dtype=float)
swap_rates_mkt_np = swap_rates_mkt.cpu().numpy()
swaps_rates_from_model = [s.fair_rate(model).detach().cpu().numpy() for s in swaps]

print("\nMaturity  Market     Model")
for m, sm, sp in zip(maturities, swap_rates_mkt_np, swaps_rates_from_model):
    print(f"{m:>7.1f}  {sm:>7.4f}  {sp.item():>7.4f}")

plt.plot(maturities, swap_rates_mkt_np, 'o-', label='Market')
plt.plot(maturities, swaps_rates_from_model, 'x--', label='Model')
plt.xlabel('Maturity (years)')
plt.ylabel('Par Swap Rate')
plt.title('Market vs Model Par Swap Rates')
plt.legend()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DeepONetDiscountCurveStable(nn.Module):
    """
    DeepONet (no ODE) that outputs discount factors with:
      DF(t) = exp(- t * softplus(g(t, curve)))
    => DF(0)=1 exactly and DF(t) in (0,1].

    Keeps the same interface for Swap.fair_rate:
      - set_curve(rates, tenors)
      - discounts(t) -> (len(t), 1)
    """
    def __init__(self, n_tenors: int, p: int = 32, hidden: int = 64, dtype=torch.float64, max_exponent: float = 60.0):
        super().__init__()
        self.n_tenors = n_tenors
        self.dtype = dtype
        self.max_exponent = max_exponent

        self.branch = nn.Sequential(
            nn.Linear(2 * n_tenors, hidden, dtype=dtype),
            nn.Tanh(),
            nn.Linear(hidden, p, dtype=dtype),
        )
        self.trunk = nn.Sequential(
            nn.Linear(1, hidden, dtype=dtype),
            nn.Tanh(),
            nn.Linear(hidden, p, dtype=dtype),
        )
        self.bias = nn.Parameter(torch.zeros((), dtype=dtype))

        self._b = None
        self._device = None
        self._dtype = None

    def set_curve(self, rates: torch.Tensor, tenors: torch.Tensor):
        if rates.ndim != 1 or tenors.ndim != 1:
            raise ValueError("rates and tenors must be 1D")
        if rates.numel() != self.n_tenors or tenors.numel() != self.n_tenors:
            raise ValueError(f"Expected n_tenors={self.n_tenors}")

        rates = rates.to(dtype=self.dtype)
        tenors = tenors.to(dtype=self.dtype)
        if rates.device != tenors.device:
            raise ValueError("rates and tenors must be on same device")

        self._device = rates.device
        self._dtype = rates.dtype

        x = torch.cat([rates, tenors], dim=0).unsqueeze(0)  # (1, 2*n_tenors)
        self._b = self.branch(x).squeeze(0)                 # (p,)

    def discounts(self, t: torch.Tensor) -> torch.Tensor:
        if self._b is None:
            raise RuntimeError("Call set_curve(rates, tenors) before pricing.")

        t = t.reshape(-1).to(device=self._device, dtype=self._dtype)  # (B,)

        h = self.trunk(t.unsqueeze(-1))                               # (B,p)
        g = (h * self._b).sum(dim=-1) + self.bias                     # (B,)
        k = F.softplus(g)                                             # (B,) >= 0
        expo = t * k                                                  # (B,) >=0        
        df = torch.exp(-expo).unsqueeze(-1)                           # (B,1) in (0,1]
        return df


In [ ]:
def train_curve_deeponet_noode_broyden(
    optimizer,
    model,
    market_rates,
    tenors_tensor,
    fit_weight: float = 100.0,
    df0_weight: float = 0.1,
):

    mse = nn.MSELoss()
    swaps = [Swap(tenor) for tenor in market_rates.columns]

    t0 = torch.tensor([0.0], dtype=tenors_tensor.dtype, device=tenors_tensor.device)
    one = torch.tensor([[1.0]], dtype=tenors_tensor.dtype, device=tenors_tensor.device)

    curves = [
        torch.tensor(row.values, dtype=tenors_tensor.dtype, device=tenors_tensor.device)
        for _, row in market_rates.iterrows()
    ]
    n = len(curves)

    def closure(*args, **kwargs):
        optimizer.zero_grad()
        total = torch.zeros((), dtype=tenors_tensor.dtype, device=tenors_tensor.device)

        for obs_rates in curves:
            model.set_curve(obs_rates, tenors_tensor)

            preds = torch.stack([s.fair_rate(model) for s in swaps])
            loss_fit = mse(preds, obs_rates)

            df0 = model.discounts(t0)
            loss_df0 = mse(df0, one)

            total = total + fit_weight * loss_fit + df0_weight * loss_df0

        total = total / max(1, n)  # <<< KEY FIX

        if not torch.isfinite(total):
            raise FloatingPointError(f"Non-finite loss: {total.item()}")

        total.backward()
        print(f"Loss: {total.item():.6e}")
        return total

    optimizer.step(closure)


In [ ]:
tenors_tensor = torch.tensor(tenors, dtype=torch.float64, device=device)  # from your earlier np tenors

model = DeepONetDiscountCurveStable(n_tenors=len(tenors), p=32, hidden=64, dtype=torch.float64).to(device)

def train_curve_deeponet_noode(
    optimizer,
    model: nn.Module,
    market_rates: "pd.DataFrame",
    tenors_tensor: torch.Tensor,   # (n_tenors,) years, torch.float64 on device
    steps: int = 10_000,
    fit_weight: float = 100.0,
    df0_weight: float = 1.0,
    print_every: int = 50,
):
    """
    Loss per curve:
      L = fit_weight * MSE(par_swap_rates_pred, par_swap_rates_obs)
        + df0_weight  * MSE(DF(0), 1)

    Uses your Swap.fair_rate (so model must provide discounts()).
    """
    mse = nn.MSELoss()

    # build swaps once
    swaps = [Swap(tenor) for tenor in market_rates.columns]

    # (constant) DF(0) target
    t0 = torch.tensor([0.0], dtype=tenors_tensor.dtype, device=tenors_tensor.device)
    one = torch.tensor([[1.0]], dtype=tenors_tensor.dtype, device=tenors_tensor.device)

    # materialize curves as tensors once (faster, less pandas overhead)
    curves = [
        torch.tensor(row.values, dtype=tenors_tensor.dtype, device=tenors_tensor.device)
        for _, row in market_rates.iterrows()
    ]

    for epoch in range(1, steps + 1):
        optimizer.zero_grad()
        total_loss = torch.zeros((), dtype=tenors_tensor.dtype, device=tenors_tensor.device)

        for obs_rates in curves:
            model.set_curve(obs_rates, tenors_tensor)

            preds = torch.stack([s.fair_rate(model) for s in swaps])  # (n_tenors,)
            loss_fit = mse(preds, obs_rates)

            df0 = model.discounts(t0)  # (1,1)
            loss_df0 = mse(df0, one)

            total_loss = total_loss + fit_weight * loss_fit + df0_weight * loss_df0

        total_loss.backward()
        optimizer.step()
        
        print(f"Epoch {epoch:5d} | Loss {total_loss.item():.6e}")

# Adam
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
train_curve_deeponet_noode(
    optimizer=opt,
    model=model,
    market_rates=market_rates_df.sample(n=50, random_state=2),
    tenors_tensor=tenors_tensor,
    steps=5000,
    fit_weight=100.0,
    df0_weight=10.0,   # usually needs to be non-trivial so DF(0) sticks to 1
    print_every=100,
)


In [ ]:
tenors_tensor = torch.tensor(tenors, dtype=torch.float64, device=device)  # from your earlier np tenors
model = DeepONetDiscountCurveStable(n_tenors=len(tenors), p=8, hidden=16, dtype=torch.float64).to(device)

In [ ]:
opt = SSBroyden(model.parameters(), max_iter=2000, tolerance_change=1e-15, tolerance_grad=1e-15)
train_curve_deeponet_noode_broyden(opt, model, market_rates_df.sample(n=200, random_state=2), tenors_tensor)

In [ ]:
# --- quick check: print market vs model --------------------------------------
# Define maturities and helper function
swap_rates_mkt = torch.tensor(market_rates_df.iloc[-1].values, dtype=torch.float64)
model.set_curve(swap_rates_mkt, tenors_tensor)

maturities = np.array([s.maturity for s in swaps], dtype=float)
swap_rates_mkt_np = swap_rates_mkt.cpu().numpy()
swaps_rates_from_model = [s.fair_rate(model).detach().cpu().numpy() for s in swaps]

print("\nMaturity  Market     Model")
for m, sm, sp in zip(maturities, swap_rates_mkt_np, swaps_rates_from_model):
    print(f"{m:>7.1f}  {sm:>7.4f}  {sp.item():>7.4f}")

plt.plot(maturities, swap_rates_mkt_np, 'o-', label='Market')
plt.plot(maturities, swaps_rates_from_model, 'x--', label='Model')
plt.xlabel('Maturity (years)')
plt.ylabel('Par Swap Rate')
plt.title('Market vs Model Par Swap Rates')
plt.legend()
plt.show()

In [ ]:
def jacobian_logdf_wrt_rates(
    model,
    rates_row: torch.Tensor,         # (n_tenors,)
    tenors_tensor: torch.Tensor,     # (n_tenors,)
    t_grid: torch.Tensor,            # (n_t,)
):
    """
    Returns:
      t_grid_cpu: (n_t,) numpy
      logdf_cpu : (n_t,) numpy
      J_cpu     : (n_t, n_tenors) numpy   where J[k,i] = d log DF(t_k) / d r_i
    """
    device = tenors_tensor.device
    dtype = tenors_tensor.dtype

    # leaf tensor so grads wrt rates work
    r = rates_row.detach().clone().to(device=device, dtype=dtype).requires_grad_(True)

    # set curve once; discounts(t_grid) will use this conditioning
    model.set_curve(r, tenors_tensor)

    # DF(t) on the grid
    df = model.discounts(t_grid.to(device=device, dtype=dtype)).squeeze(-1)  # (n_t,)
    if torch.any(df <= 0):
        raise FloatingPointError("DF(t) <= 0 encountered; log is invalid.")

    logdf = torch.log(df)  # (n_t,)

    n_t = logdf.numel()
    n_r = r.numel()
    J = torch.empty((n_t, n_r), dtype=dtype, device=device)

    # Compute each row of the Jacobian via autograd.grad
    for k in range(n_t):
        (g,) = torch.autograd.grad(
            outputs=logdf[k],
            inputs=r,
            retain_graph=True,
            create_graph=False,
            allow_unused=False,
        )
        J[k, :] = g

    return (
        t_grid.detach().cpu().numpy(),
        logdf.detach().cpu().numpy(),
        J.detach().cpu().numpy(),
    )

def plot_jacobian_heatmap(
    t_grid_cpu: np.ndarray,          # (n_t,)
    J_cpu: np.ndarray,               # (n_t, n_tenors)
    pillar_labels=None,              # list[str] length n_tenors (e.g. market_rates_df.columns)
    title="Jacobian heatmap: d log DF(t) / d r_i",
):
    plt.figure(figsize=(10, 6))
    im = plt.imshow(
        J_cpu,
        aspect="auto",
        origin="lower",
        extent=[0, J_cpu.shape[1], t_grid_cpu[0], t_grid_cpu[-1]],
    )
    plt.colorbar(im, label=r"$\partial \log DF(t) / \partial r_i$")
    plt.xlabel("Pillar index i")
    plt.ylabel("Maturity t (years)")
    plt.title(title)

    if pillar_labels is not None:
        # show a readable subset of labels
        n = len(pillar_labels)
        max_ticks = 12
        idx = np.linspace(0, n - 1, min(n, max_ticks), dtype=int)
        plt.xticks(idx + 0.5, [pillar_labels[i] for i in idx], rotation=45, ha="right")

    plt.tight_layout()
    plt.show()


In [ ]:
# pick one curve (e.g. last date)
rates_row = torch.tensor(market_rates_df.iloc[-1].values, dtype=tenors_tensor.dtype, device=tenors_tensor.device)

# maturity grid for the heatmap
t_grid = torch.linspace(0.0, 10.0, 241, dtype=tenors_tensor.dtype, device=tenors_tensor.device)
t_grid[0] = 1e-6  # avoid log(DF(0)) corner if you have exact DF(0)=1; keeps gradients well-behaved

t_cpu, logdf_cpu, J_cpu = jacobian_logdf_wrt_rates(model, rates_row, tenors_tensor, t_grid)

plot_jacobian_heatmap(
    t_cpu,
    J_cpu,
    pillar_labels=list(market_rates_df.columns),
    title="d log DF(t) / d (swap pillar rates)",
)

plt.figure()
plt.plot(t_cpu, logdf_cpu)
plt.xlabel("Maturity t (years)")
plt.ylabel("log DF(t)")
plt.title("log Discount Factor curve")
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import plotly.graph_objects as go

def plot_jacobian_surface_plotly(
    t_grid_cpu: np.ndarray,      # (n_t,)
    J_cpu: np.ndarray,           # (n_t, n_tenors)
    pillar_labels=None,          # optional list[str]
    stride_t: int = 2,
    stride_i: int = 1,
    title: str = "Surface: d log DF(t) / d r_i",
):
    """
    Plotly 3D surface:
      x = pillar index (or labels)
      y = maturity t (years)
      z = J(t,i)

    Notes:
      - Plotly Surface expects z with shape (len(y), len(x)).
      - If you pass labels for x, Plotly will use categorical axis.
    """
    # downsample for performance
    t = t_grid_cpu[::stride_t]
    Z = J_cpu[::stride_t, ::stride_i]

    n_i = Z.shape[1]
    x_idx = np.arange(n_i)

    if pillar_labels is not None:
        full_labels = list(pillar_labels)[::stride_i]
        # Plotly will treat these as categories
        x = full_labels
        x_title = "Pillar tenor"
    else:
        x = x_idx
        x_title = "Pillar index i"

    fig = go.Figure(
        data=go.Surface(
            x=x,
            y=t,
            z=Z,
            colorbar=dict(title="d log DF / d r"),
        )
    )

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(title=x_title),
            yaxis=dict(title="Maturity t (years)"),
            zaxis=dict(title="d log DF / d r"),
        ),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    fig.show()

plot_jacobian_surface_plotly(
    t_grid_cpu=t_cpu,
    J_cpu=J_cpu,
    pillar_labels=list(market_rates_df.columns),  # or None for numeric indices
    stride_t=2,
    stride_i=1,
    title="Jacobian surface: d log DF(t) / d swap pillar rates",
)


In [ ]:
import torch
import numpy as np

def jacobian_swaprates_wrt_pillarrates(
    model,
    rates_row: torch.Tensor,         # (n_tenors,)
    tenors_tensor: torch.Tensor,     # (n_tenors,)
    swap_tenor_labels,               # list of tenors (same used to build swaps)
    eps: float = 1e-12,
):
    """
    Returns:
      preds_cpu : (n_swaps,) numpy
      J_cpu     : (n_swaps, n_tenors) numpy, J[j,i] = d S_pred[j] / d rates_row[i]
    """
    device = tenors_tensor.device
    dtype = tenors_tensor.dtype

    # leaf tensor for grads wrt input rates
    r = rates_row.detach().clone().to(device=device, dtype=dtype).requires_grad_(True)

    # Build swaps once
    swaps = [Swap(tenor) for tenor in swap_tenor_labels]
    # Optional: ensure Swap.fair_rate uses eps in PV01 denom
    # (if you already modified it, ignore)
    def fair_rate_safe(swap):
        return swap.fair_rate(model)

    # Set curve conditioning
    # For your direct DeepONet DF model:
    model.set_curve(r, tenors_tensor)

    # Vector of predicted par swap rates (n_swaps,)
    preds = torch.stack([fair_rate_safe(s) for s in swaps])

    n_swaps = preds.numel()
    n_tenors = r.numel()
    J = torch.empty((n_swaps, n_tenors), dtype=dtype, device=device)

    # Row-by-row grads: d preds[j] / d r
    for j in range(n_swaps):
        (g,) = torch.autograd.grad(
            outputs=preds[j],
            inputs=r,
            retain_graph=True,
            create_graph=False,
            allow_unused=False,
        )
        J[j, :] = g

    return preds.detach().cpu().numpy(), J.detach().cpu().numpy()



import numpy as np
import plotly.graph_objects as go

def plot_swaprate_sensitivity_surface_plotly_fit(
    J_cpu: np.ndarray,               # (n_out, n_in)
    in_labels,                       # list[str] length n_in
    out_labels,                      # list[str] length n_out
    title: str = "Surface: d S_out / d r_in",
    stride_out: int = 1,
    stride_in: int = 1,
    width: int = 1100,
    height: int = 800,
    margin_l: int = 80,
    margin_r: int = 40,
    margin_t: int = 70,
    margin_b: int = 120,
    max_ticks: int = 10,
):
    """
    Uses numeric x/y for the surface to avoid clipping, and adds readable tick labels.
    Tenor labels still show in hover via customdata.
    """
    Z = J_cpu[::stride_out, ::stride_in]
    in_labels_ds = list(in_labels)[::stride_in]
    out_labels_ds = list(out_labels)[::stride_out]

    n_out, n_in = Z.shape
    x = np.arange(n_in)
    y = np.arange(n_out)

    # customdata will carry (out_tenor, in_tenor)
    customdata = np.empty((n_out, n_in, 2), dtype=object)
    for j in range(n_out):
        for i in range(n_in):
            customdata[j, i, 0] = out_labels_ds[j]
            customdata[j, i, 1] = in_labels_ds[i]

    # Choose tick positions (limit clutter)
    def _tickvals_text(labels, n, max_ticks):
        if n <= max_ticks:
            vals = np.arange(n)
        else:
            vals = np.linspace(0, n - 1, max_ticks, dtype=int)
        text = [labels[v] for v in vals]
        return vals.tolist(), text

    xtickvals, xticktext = _tickvals_text(in_labels_ds, n_in, max_ticks)
    ytickvals, yticktext = _tickvals_text(out_labels_ds, n_out, max_ticks)

    fig = go.Figure(
        data=go.Surface(
            x=x,
            y=y,
            z=Z,
            customdata=customdata,
            hovertemplate=(
                "Output swap: %{customdata[0]}<br>"
                "Input pillar: %{customdata[1]}<br>"
                "dS/dr: %{z:.6g}<extra></extra>"
            ),
            colorbar=dict(title="dS/dr"),
        )
    )

    fig.update_layout(
        title=title,
        autosize=False,
        width=width,
        height=height,
        margin=dict(l=margin_l, r=margin_r, t=margin_t, b=margin_b),
        scene=dict(
            xaxis=dict(
                title="Input pillar (r_in)",
                tickmode="array",
                tickvals=xtickvals,
                ticktext=xticktext,
                tickangle=-35,
            ),
            yaxis=dict(
                title="Output swap (S_out)",
                tickmode="array",
                tickvals=ytickvals,
                ticktext=yticktext,
                tickangle=-20,
            ),
            zaxis=dict(title="d S_out / d r_in"),
            aspectmode="auto",  # or "cube" / "data" depending on preference
        ),
    )

    fig.show()


In [ ]:
import torch
import torch.nn as nn

class SwapMaturity:
    """
    Swap defined by maturity in years (float).
    Payment frequency:
      - if maturity < 2y -> single payment at maturity (like your current logic)
      - else -> semi-annual
    """
    def __init__(self, maturity_years: float, dtype=torch.float64, device="cpu"):
        self.maturity = float(maturity_years)
        self.dtype = dtype
        self.device = device

        if self.maturity < 2.0:
            self.frequency = self.maturity
        else:
            self.frequency = 0.5

        # cashflow times including 0
        self.yfs = torch.arange(
            0.0, self.maturity + self.frequency + 1e-12, self.frequency,
            dtype=self.dtype, device=self.device
        )

    def fair_rate(self, model: nn.Module, eps: float = 1e-12) -> torch.Tensor:
        dfs = model.discounts(self.yfs).squeeze(-1)          # (n_times,)
        dfs = torch.clamp(dfs, min=eps, max=1.0)

        pay_dfs = dfs[1:]                                   # exclude t=0
        pv01 = torch.sum(pay_dfs * self.frequency) + eps
        fixed_leg = 1.0 - pay_dfs[-1]
        return (fixed_leg / pv01).squeeze()


In [ ]:
import numpy as np
import torch

def pillar_years_from_columns(columns) -> np.ndarray:
    # uses your tenor_to_years() defined earlier (handles D/W/M/Y)
    return np.array([tenor_to_years(c) for c in columns], dtype=float)

def half_year_day_tenors(max_years: float, step_years: float = 0.5):
    days_step = int(round(step_years * 360))  # because Swap uses days/360
    n = int(round(max_years / step_years))
    return [f"{k * days_step}d" for k in range(1, n + 1)]


def jacobian_swap_surface_wrt_inputpillars(
    model,
    rates_row: torch.Tensor,          # (n_in,) observed pillar rates for one date
    tenors_in_years: torch.Tensor,    # (n_in,) years corresponding to input pillars
    out_tenor_labels: list[str],      # e.g. ['180d','360d',...]
):
    """
    Returns:
      T_out_years : (n_out,) numpy
      preds_cpu   : (n_out,) numpy
      J_cpu       : (n_out, n_in) numpy where J[k,i]=d S(T_out[k]) / d r_in[i]
    """
    device = tenors_in_years.device
    dtype = tenors_in_years.dtype

    # leaf tensor for gradients
    r = rates_row.detach().clone().to(device=device, dtype=dtype).requires_grad_(True)

    swaps_out = [Swap(t) for t in out_tenor_labels]
    T_out_years = np.array([s.maturity for s in swaps_out], dtype=float)

    # condition on full input curve (your actual pillars)
    model.set_curve(r, tenors_in_years)

    preds = torch.stack([s.fair_rate(model) for s in swaps_out])  # (n_out,)

    n_out = preds.numel()
    n_in = r.numel()
    J = torch.empty((n_out, n_in), dtype=dtype, device=device)

    for k in range(n_out):
        (g,) = torch.autograd.grad(
            outputs=preds[k],
            inputs=r,
            retain_graph=True,
            create_graph=False,
            allow_unused=False,
        )
        J[k, :] = g

    return T_out_years, preds.detach().cpu().numpy(), J.detach().cpu().numpy()

import plotly.graph_objects as go

def plot_surface_years_years_plotly(
    x_years: np.ndarray,   # (n_in,) input pillar maturities
    y_years: np.ndarray,   # (n_out,) output maturities (0.5y grid)
    Z: np.ndarray,         # (n_out, n_in)
    title="Surface: d S(T_out) / d r_in",
    width=1200,
    height=850,
):
    fig = go.Figure(
        data=go.Surface(
            x=x_years,
            y=y_years,
            z=Z,
            colorbar=dict(title="dS/dr"),
            contours=dict(z=dict(show=True, usecolormap=True, project_z=True)),
        )
    )
    fig.update_layout(
        title=title,
        width=width,
        height=height,
        margin=dict(l=80, r=40, t=70, b=70),
        scene=dict(
            xaxis=dict(title="Input pillar maturity T_in (years)"),
            yaxis=dict(title="Output swap maturity T_out (years)"),
            zaxis=dict(title="d S(T_out) / d r_in"),
            aspectmode="auto",
        ),
    )
    fig.show()

# pick a date row
row = market_rates_df.iloc[-1]
rates_row = torch.tensor(row.values, dtype=dtype, device=device)

# input pillar maturities from your REAL column names (e.g. '6M','1Y',...)
x_in_years = pillar_years_from_columns(market_rates_df.columns)
tenors_in_years = torch.tensor(x_in_years, dtype=dtype, device=device)

# output maturities: 0.5y grid up to (say) 20y
out_labels = half_year_day_tenors(max_years=10.0, step_years=0.5)

T_out_years, preds_cpu, J_cpu = jacobian_swap_surface_wrt_inputpillars(
    model=model,
    rates_row=rates_row,
    tenors_in_years=tenors_in_years,
    out_tenor_labels=out_labels,
)

plot_surface_years_years_plotly(
    x_years=x_in_years,
    y_years=T_out_years,
    Z=J_cpu,
    title="Surface: d (model par swap rate at T_out) / d (input pillar rate)",
)
